# 04 — Hybrid Search: vector + filters + BM25

**Talk role:** Demonstrates what a real search engine does that a pure vector DB structurally can't. The slide where you explain why Elastic-over-Mongo matters for multimodal work.

Three queries that aren't possible with kNN alone:
1. "Outdoor shot under 15 seconds uploaded this week"
2. "Clip about coffee, but from uploader X, with tag 'product'"
3. "Person laughing" — fused with BM25 over the transcript field

In [ ]:
import sys, datetime as dt
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'scripts'))

from embed_jina import JinaClient, EmbedConfig
from index_elastic import es_client, knn_search, hybrid_search, index_name

jina = JinaClient()
es = es_client()
INDEX = index_name('scene', 1024, 'float')

def embed_q(text):
    return jina.embed([text], task='retrieval.query', config=EmbedConfig())[0]

## Demo 1: Pure kNN vs. kNN + filter

The same query, the same embedding, different filters. Notice how the result set narrows in a way that's *correct*, not just smaller.

In [ ]:
qvec = embed_q('outdoor establishing shot')

print('— Pure kNN —')
for h in knn_search(es, INDEX, qvec, k=5):
    print(f'  {h["_score"]:.3f}  {h["chunk_id"]:45s}  {h["duration"]:5.1f}s  {h.get("uploaded_at", "")}')

print('\n— + duration <= 15s, uploaded last 7 days —')
week_ago = (dt.date.today() - dt.timedelta(days=7)).isoformat()
filters = [
    {'range': {'duration': {'lte': 15}}},
    {'range': {'uploaded_at': {'gte': week_ago}}},
]
for h in knn_search(es, INDEX, qvec, k=5, filter_clauses=filters):
    print(f'  {h["_score"]:.3f}  {h["chunk_id"]:45s}  {h["duration"]:5.1f}s  {h.get("uploaded_at", "")}')

## Demo 2: RRF fusion of BM25 (transcript) and vector (visual)

Why this matters: for clips where someone is talking, the visual embedding alone may miss specific terminology. BM25 over the transcript catches it. RRF combines both into one ranking without either side dominating.

In [ ]:
QUERY = 'product demo Elasticsearch dashboard'
qvec = embed_q(QUERY)

print('— Vector only —')
for h in knn_search(es, INDEX, qvec, k=5):
    print(f'  {h["_score"]:.3f}  {h["chunk_id"]:45s}')

print('\n— Hybrid (RRF: vector + BM25 over transcript) —')
for h in hybrid_search(es, INDEX, QUERY, qvec, k=5):
    print(f'  {h["_score"]:.3f}  {h["chunk_id"]:45s}')

## Demo 3: Aggregations — "what kinds of clips have I uploaded this month?"

Pure-vector DBs can't do this. Elastic gives you facets + top-N + date histograms over the *same* result set.

In [ ]:
res = es.search(index=INDEX, size=0, aggs={
    'by_uploader': {'terms': {'field': 'uploader', 'size': 10}},
    'by_tag':      {'terms': {'field': 'tags',     'size': 10}},
    'by_week':     {'date_histogram': {'field': 'uploaded_at', 'calendar_interval': 'week'}},
    'avg_dur':     {'avg': {'field': 'duration'}},
})
print('Uploaders:')
for b in res['aggregations']['by_uploader']['buckets']:
    print(f'  {b["key"]:20s}  {b["doc_count"]} chunks')
print(f'\nAvg chunk duration: {res["aggregations"]["avg_dur"]["value"]:.1f}s')